# Alignment pipeline (debug, fixed)

Robust discovery of episodes_audio by scanning upward through parent directories. Prints cwd, chosen episodes dir and found .webm files.

In [7]:
!git clone https://github.com/Yi-Star32/Video_Analyzer.git

Cloning into 'Video_Analyzer'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 39 (delta 4), reused 29 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 52.41 KiB | 4.76 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [8]:
%cd Video_Analyzer

/content/Video_Analyzer/Video_Analyzer


In [9]:
!pip install -r requirements.txt

  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached asttokens-3.0.1-py3-none-any.whl.metadata (4.9 kB)
ERROR: Ignored the following versions that require a different python version: 0.1.0 Requires-Python >=3.13; 0.2.0 Requires-Python >=3.13; 0.2.1 Requires-Python >=3.13; 0.2.2 Requires-Python >=3.13
ERROR: Could not find a version that satisfies the requirement audioop-lts==0.2.2 (from versions: none)
ERROR: No matching distribution found for audioop-lts==0.2.2


In [10]:
!pip install faiss-cpu

In [11]:
!pip install git+https://github.com/m-bain/whisperx.git

  Cloning https://github.com/m-bain/whisperx.git to /tmp/pip-req-build-5v6lag1c
  Running command git clone --filter=blob:none --quiet https://github.com/m-bain/whisperx.git /tmp/pip-req-build-5v6lag1c
  Resolved https://github.com/m-bain/whisperx.git to commit 3ccc17b8de34f305300f8a3fd3c9f76ba820c0d0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==

In [12]:
!pip install "numpy<2.0.0"

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
whisperx 3.8.6 requires numpy>=2.1.0, but you have numpy 1.26.4 which is incompatible.
pyannote-metrics 4.1 requires numpy>=2.2.2, but you have numpy 1.26.4 which is incompatible.
pyannote-core 6.0.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, bu

In [13]:
from audio_matcher import alignment, chunking, embedding, index, io, phonemes

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import sys
from pathlib import Path
import warnings
from tqdm import TqdmWarning

# 1. Zorg dat Python de 'audio_matcher' map kan vinden
# (Pas dit pad aan als je Video_Analyzer map ergens anders staat, bv. in je Drive)
project_path = "/content/Video_Analyzer"
if project_path not in sys.path:
    sys.path.append(project_path)

# 2. Onderdruk specifieke tqdm waarschuwingen
warnings.filterwarnings("ignore", category=TqdmWarning)

# 3. Importeer de audio_matcher onderdelen
from audio_matcher.embedding import AudioEmbeddingPipeline
from audio_matcher.phonemes import PhonemeAligner
from audio_matcher.alignment import build_phoneme_index_from_episodes, run_phoneme_pipeline
from audio_matcher.io import export_audio

# 4. Stel het absolute pad in naar je song (zorg dat je Drive gekoppeld is!)
SONG_PATH = Path("/content/drive/MyDrive/projecten/Video_Analyzer_data/output/separated/htdemucs/audio/vocals.wav")

# 5. Check direct of het bestand bestaat
if SONG_PATH.exists():
    print("🎉 Alles succesvol geïmporteerd en het audiobestand is gevonden!")
else:
    print("⚠️ Imports geslaagd, maar het bestand 'vocals.wav' werd niet gevonden op dit pad. Check of je Drive is gekoppeld.")

🎉 Alles succesvol geïmporteerd en het audiobestand is gevonden!


In [16]:
from pathlib import Path

# Pas dit aan naar de exacte plek van je projectmap
# Als hij in je Drive staat: Path("/content/drive/MyDrive/.../Video_Analyzer")
repo_root = Path("/content/drive/MyDrive/projecten/Video_Analyzer_data")

print('notebook cwd ingesteld op =', repo_root)
EPISODES_DIR = None

# Vanaf hier blijft je eigen code exact hetzelfde:
for p in [repo_root] + list(repo_root.parents):
    candidates = [
        p / 'data' / 'episodes_audio',
        p / 'audio_matcher' / 'data' / 'episodes_audio',
        p / 'episodes_audio',
    ]
    for c in candidates:
        if c.exists():
            EPISODES_DIR = c
            break
    if EPISODES_DIR is not None:
        break

if EPISODES_DIR is None:
    EPISODES_DIR = repo_root / 'data' / 'episodes_audio'

print('EPISODES_DIR chosen:', EPISODES_DIR)
if not EPISODES_DIR.exists():
    print('EPISODES_DIR does not exist:', EPISODES_DIR)
    files = []
else:
    files = sorted(EPISODES_DIR.rglob('audio.wav'))
    print(f'Found {len(files)} audio.wav files under {EPISODES_DIR}')
    for f in files:
        print('-', f)

notebook cwd ingesteld op = /content/drive/MyDrive/projecten/Video_Analyzer_data
EPISODES_DIR chosen: /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio
Found 30 audio.wav files under /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/1TlOcjJodHw/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/326R_Lhua5w/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/3l1lSNQxJA0/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/6uNYmqvF24k/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/9IymXcXl4fk/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/9aLD8SGoPIc/audio.wav
- /content/drive/MyDrive/projecten/Video_Analyzer_data/data/episodes_audio/9wAKxOavTnw/audio.wav
- /content/drive/MyDrive/projecten/Video_A

In [17]:
import gc
import torch
import numpy as np

# Automatisch GPU kiezen als die er is, anders veilig terugvallen op CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 WhisperX gaat draaien op: {device.upper()}")

pipeline = AudioEmbeddingPipeline()

# Hier vullen we de 'device' variabele in
aligner = PhonemeAligner(device=device, whisper_model='base')

# Je lijst met bestanden uit de vorige cell
files = files

# Start het bouwen van de index
pindex = build_phoneme_index_from_episodes(files, aligner, pipeline)

print("🎉 De phoneme index is succesvol opgebouwd!")

🚀 WhisperX gaat draaien op: CUDA


/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Building phoneme index from episodes:   0%|          | 0/30 [00:00<?, ?it/s]

2026-06-02 11:41:31 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-06-02 11:41:31 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`


2026-06-02 11:41:38 - whisperx.asr - INFO - Detected language: en (0.81) in first 30s of audio
  [audio.wav] 8994 phonemes


Building phoneme index from episodes:   0%|          | 0/30 [00:59<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# final: run phoneme pipeline on chosen song and export
if not files:
    print('No reference files found, skipping pipeline')
else:
    final_audio = run_phoneme_pipeline(SONG_PATH, None, aligner, pipeline, pindex=pindex)
    print(f"output_ms: {len(final_audio)}")
    export_audio(final_audio, '/content/drive/MyDrive/projecten/Video_Analyzer_data/output/aligned_output_colab1.wav')
    print('Wrote aligned_output.wav')
